# 05_test_hybrid_retrieval

## 목적

이 노트북은 `04_build_vector_bm25_index.ipynb`에서 생성한 구조화 검색 인덱스를 불러와
실제 hybrid retrieval 품질을 테스트한다.

입력:
- data/chromadb/
- Chroma collection: complypilot_regulations_v2
- data/retrieval/bm25_index/bm25.pkl
- data/retrieval/parents.jsonl
- data/retrieval/children.jsonl

이번 노트북에서 하는 일:

1. ChromaDB collection 로드
2. BM25 index 로드
3. parents.jsonl, children.jsonl 로드
4. query_builder 구현
5. vector_search 구현
6. bm25_search 구현
7. RRF merge 구현
8. document_priority, risk_tag, keyword match 기반 deterministic rerank 구현
9. parent_id 기준 dedupe
10. parent context expansion
11. report-safe evidence format 생성
12. seed query 기반 검색 품질 확인

이번 노트북에서는 아직 LangGraph evidence_retriever node에 연결하지 않는다.
이번 노트북에서는 아직 Streamlit report를 수정하지 않는다.

In [1]:
# 기본 import / 경로 설정
from pathlib import Path
import os
import re
import json
import pickle
from pprint import pprint
from typing import List, Dict, Any, Tuple
from collections import defaultdict, Counter

import numpy as np
import pandas as pd

from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma


CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

env_path = PROJECT_ROOT / ".env"
if env_path.exists():
    load_dotenv(env_path, override=True)

RETRIEVAL_DIR = PROJECT_ROOT / "data" / "retrieval"
CHROMA_DB_DIR = PROJECT_ROOT / "data" / "chromadb"
BM25_DIR = RETRIEVAL_DIR / "bm25_index"
DEBUG_DIR = RETRIEVAL_DIR / "debug_hybrid_retrieval"

PARENTS_PATH = RETRIEVAL_DIR / "parents.jsonl"
CHILDREN_PATH = RETRIEVAL_DIR / "children.jsonl"
BM25_PATH = BM25_DIR / "bm25.pkl"
SUMMARY_PATH = DEBUG_DIR / "hybrid_retrieval_summary.json"

DEBUG_DIR.mkdir(parents=True, exist_ok=True)

COLLECTION_NAME = "complypilot_regulations_v2"
EMBEDDING_MODEL_NAME = "text-embedding-3-small"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CHROMA_DB_DIR:", CHROMA_DB_DIR)
print("PARENTS_PATH:", PARENTS_PATH)
print("CHILDREN_PATH:", CHILDREN_PATH)
print("BM25_PATH:", BM25_PATH)
print("COLLECTION_NAME:", COLLECTION_NAME)
print("OPENAI_API_KEY exists:", bool(os.getenv("OPENAI_API_KEY")))

c:\Users\USER\Desktop\complypilot-jb\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PROJECT_ROOT: c:\Users\USER\Desktop\complypilot-jb
CHROMA_DB_DIR: c:\Users\USER\Desktop\complypilot-jb\data\chromadb
PARENTS_PATH: c:\Users\USER\Desktop\complypilot-jb\data\retrieval\parents.jsonl
CHILDREN_PATH: c:\Users\USER\Desktop\complypilot-jb\data\retrieval\children.jsonl
BM25_PATH: c:\Users\USER\Desktop\complypilot-jb\data\retrieval\bm25_index\bm25.pkl
COLLECTION_NAME: complypilot_regulations_v2
OPENAI_API_KEY exists: True


In [2]:
# JSONL 로드 함수
def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    """
    JSONL 파일을 읽어 dict 리스트로 반환합니다.

    Args:
        path: JSONL 파일 경로

    Return:
        dict 리스트
    """
    rows = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))

    return rows


parents = load_jsonl(PARENTS_PATH)
children = load_jsonl(CHILDREN_PATH)

parent_map = {
    row["parent_id"]: row
    for row in parents
    if row.get("parent_id")
}

print("parents:", len(parents))
print("children:", len(children))
print("parent_map:", len(parent_map))

pprint(children[0] if children else None)

parents: 608
children: 2155
parent_map: 608
{'article_no': '제1조',
 'article_title': '목적',
 'child_id': 'financial_consumer_supervisory_regulation__article_1__child_001_ef419dd8',
 'child_index': 1,
 'child_status': 'ok',
 'child_text': '이 규정은 「금융소비자 보호에 관한 법률」 및 같은 법 시행령에서 위임하는 사항과 그 시행에 필요한\n'
               '사항을 규정함을 목적으로 한다.',
 'chunk_id': 'financial_consumer_supervisory_regulation__article_1__child_001_ef419dd8',
 'doc_code': 'financial_consumer_supervisory_regulation',
 'document_priority': 3,
 'document_type': 'supervisory_regulation',
 'effective_date': '2026.4.2.',
 'has_article_no': True,
 'has_page': True,
 'has_parent_id': True,
 'is_long_child': False,
 'is_short_child': False,
 'item_no': '',
 'keywords': [],
 'law_name': '금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402)',
 'page': 1,
 'page_end': 1,
 'page_start': 1,
 'paragraph_no': '',
 'parent_id': 'financial_consumer_supervisory_regulation__article_1',
 'parent_text': '제1조(목적) 이 규정은 「금융소비자 보호에 관한 법률」 및 같은 법 시행령에서 위임하는 

In [3]:
# BM25 index 로드
with open(BM25_PATH, "rb") as f:
    bm25_payload = pickle.load(f)

bm25 = bm25_payload["bm25"]
bm25_ids = bm25_payload["ids"]
bm25_documents = bm25_payload["documents"]
bm25_metadatas = bm25_payload["metadatas"]

print("BM25 payload keys:", bm25_payload.keys())
print("BM25 ids:", len(bm25_ids))
print("BM25 documents:", len(bm25_documents))
print("BM25 metadatas:", len(bm25_metadatas))
print("BM25 collection_name:", bm25_payload.get("collection_name"))

assert bm25_payload.get("collection_name") == COLLECTION_NAME
assert len(bm25_ids) == len(bm25_documents) == len(bm25_metadatas)

print("[OK] BM25 index 로드 완료")

BM25 payload keys: dict_keys(['bm25', 'ids', 'documents', 'metadatas', 'children', 'tokenizer', 'collection_name', 'embedding_provider', 'embedding_model'])
BM25 ids: 2155
BM25 documents: 2155
BM25 metadatas: 2155
BM25 collection_name: complypilot_regulations_v2
[OK] BM25 index 로드 완료


In [4]:
# Chroma vectorstore 로드
embedding_model = OpenAIEmbeddings(
    model=EMBEDDING_MODEL_NAME
)

vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embedding_model,
    persist_directory=str(CHROMA_DB_DIR),
)

raw_collection = vectorstore._collection
chroma_count = raw_collection.count()

print("Chroma collection:", COLLECTION_NAME)
print("Chroma count:", chroma_count)

assert chroma_count == len(children), "Chroma count와 children 수가 다릅니다."

print("[OK] Chroma vectorstore 로드 완료")

Chroma collection: complypilot_regulations_v2
Chroma count: 2155
[OK] Chroma vectorstore 로드 완료


In [5]:
# 기본 유틸 함수
def split_pipe_string(value: Any) -> List[str]:
    """
    pipe(|)로 저장된 metadata 문자열을 list로 변환합니다.

    Args:
        value: metadata 값

    Return:
        문자열 리스트
    """
    if value is None:
        return []

    if isinstance(value, list):
        return [str(x) for x in value if str(x)]

    value = str(value)

    if not value:
        return []

    return [x for x in value.split("|") if x]


def tokenize_for_bm25(text: str) -> List[str]:
    """
    BM25 검색용 간단 토큰화를 수행합니다.

    Args:
        text: 입력 텍스트

    Return:
        토큰 리스트
    """
    text = text.lower()
    tokens = re.findall(r"[가-힣A-Za-z0-9]+", text)
    tokens = [token for token in tokens if len(token) >= 2]
    return tokens


def normalize_text(text: str) -> str:
    """
    비교용 텍스트를 정규화합니다.

    Args:
        text: 입력 텍스트

    Return:
        정규화된 텍스트
    """
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def safe_float(value: Any, default: float = 0.0) -> float:
    """
    값을 float로 안전하게 변환합니다.

    Args:
        value: 입력 값
        default: 실패 시 기본값

    Return:
        float 값
    """
    try:
        return float(value)
    except Exception:
        return default

In [6]:
# Query profile / query builder
RISK_QUERY_RULES = {
    "approval_misleading": {
        "triggers": ["누구나 승인", "무조건 승인", "100% 승인", "승인 보장", "누구나", "무조건"],
        "expanded_query": "승인 가능성 오인 금융상품 광고 소비자 오인 조건 누구에게나 적용 승인 보장",
        "preferred_risk_tags": ["approval_misleading", "advertising_regulation"],
        "preferred_keywords": ["승인", "광고", "오인", "조건", "보장"],
    },
    "rate_condition_missing": {
        "triggers": ["최저금리", "금리", "이자율", "우대금리"],
        "expanded_query": "금리 이자율 최저금리 조건 우대금리 광고 오인 중요사항 고지 설명의무",
        "preferred_risk_tags": ["rate_condition_missing", "advertising_regulation", "explanation_duty"],
        "preferred_keywords": ["금리", "이자율", "광고", "고지", "조건"],
    },
    "fee_missing": {
        "triggers": ["수수료", "비용", "중도상환", "연체", "부대비용"],
        "expanded_query": "수수료 비용 부대비용 중도상환수수료 연체이자 고지 설명의무 금융상품 중요사항",
        "preferred_risk_tags": ["fee_missing", "explanation_duty"],
        "preferred_keywords": ["수수료", "비용", "고지", "설명", "중도상환"],
    },
    "principal_guarantee_misleading": {
        "triggers": ["원금보장", "원금 보장", "손실없음", "손실 없음"],
        "expanded_query": "원금 손실 보장 오인 투자성 상품 광고 위험 고지 설명의무",
        "preferred_risk_tags": ["principal_guarantee_misleading", "advertising_regulation", "explanation_duty"],
        "preferred_keywords": ["원금", "손실", "보장", "위험", "광고"],
    },
    "return_misleading": {
        "triggers": ["확정수익", "고수익", "수익보장", "수익률 보장"],
        "expanded_query": "수익 수익률 확정수익 보장 오인 광고 투자 위험 고지 설명의무",
        "preferred_risk_tags": ["return_misleading", "advertising_regulation", "explanation_duty"],
        "preferred_keywords": ["수익", "수익률", "보장", "오인", "광고"],
    },
    "explanation_duty": {
        "triggers": ["설명의무", "설명 의무", "중요사항", "고지"],
        "expanded_query": "설명의무 중요사항 고지 금융상품 소비자 설명 금융상품판매업자",
        "preferred_risk_tags": ["explanation_duty"],
        "preferred_keywords": ["설명", "설명의무", "중요사항", "고지"],
    },
    "unfair_solicitation": {
        "triggers": ["부당권유", "권유", "적합성", "적정성"],
        "expanded_query": "부당권유 권유 적합성 적정성 금융소비자 보호 판매 규제",
        "preferred_risk_tags": ["unfair_solicitation"],
        "preferred_keywords": ["부당권유", "권유", "적합성", "적정성"],
    },
    "advertising_regulation": {
        "triggers": ["광고", "오인", "과장", "표시"],
        "expanded_query": "금융상품 광고 표시 오인 과장 광고 금지행위 소비자 보호",
        "preferred_risk_tags": ["advertising_regulation"],
        "preferred_keywords": ["광고", "표시", "오인", "과장"],
    },
}


def build_query_profile(query: str) -> Dict[str, Any]:
    """
    원문 질의를 retrieval에 적합한 query profile로 변환합니다.

    Args:
        query: 사용자 질의 또는 위험 문구

    Return:
        query profile dict
    """
    matched_risk_types = []
    expanded_parts = [query]
    preferred_risk_tags = []
    preferred_keywords = []

    for risk_type, rule in RISK_QUERY_RULES.items():
        if any(trigger in query for trigger in rule["triggers"]):
            matched_risk_types.append(risk_type)
            expanded_parts.append(rule["expanded_query"])
            preferred_risk_tags.extend(rule["preferred_risk_tags"])
            preferred_keywords.extend(rule["preferred_keywords"])

    if not matched_risk_types:
        matched_risk_types = ["general"]
        expanded_parts.append("금융상품 광고 소비자 오인 중요사항 고지 설명의무")
        preferred_risk_tags.extend(["advertising_regulation", "explanation_duty"])
        preferred_keywords.extend(tokenize_for_bm25(query))

    expanded_query = " ".join(expanded_parts)

    return {
        "original_query": query,
        "expanded_query": expanded_query,
        "matched_risk_types": sorted(set(matched_risk_types)),
        "preferred_risk_tags": sorted(set(preferred_risk_tags)),
        "preferred_keywords": sorted(set(preferred_keywords)),
        "preferred_document_types": [
            "law",
            "enforcement_decree",
            "supervisory_regulation",
            "guideline",
            "faq_or_manual",
        ],
    }


sample_profiles = [
    build_query_profile("누구나 승인"),
    build_query_profile("최저금리"),
    build_query_profile("수수료"),
]

pprint(sample_profiles)

[{'expanded_query': '누구나 승인 승인 가능성 오인 금융상품 광고 소비자 오인 조건 누구에게나 적용 승인 보장',
  'matched_risk_types': ['approval_misleading'],
  'original_query': '누구나 승인',
  'preferred_document_types': ['law',
                               'enforcement_decree',
                               'supervisory_regulation',
                               'guideline',
                               'faq_or_manual'],
  'preferred_keywords': ['광고', '보장', '승인', '오인', '조건'],
  'preferred_risk_tags': ['advertising_regulation', 'approval_misleading']},
 {'expanded_query': '최저금리 금리 이자율 최저금리 조건 우대금리 광고 오인 중요사항 고지 설명의무',
  'matched_risk_types': ['rate_condition_missing'],
  'original_query': '최저금리',
  'preferred_document_types': ['law',
                               'enforcement_decree',
                               'supervisory_regulation',
                               'guideline',
                               'faq_or_manual'],
  'preferred_keywords': ['고지', '광고', '금리', '이자율', '조건'],
  'preferred_risk_tags': ['ad

In [7]:
# Vector search
def vector_search(
    query_profile: Dict[str, Any],
    top_k: int = 10,
    filter_dict: Dict[str, Any] | None = None,
) -> List[Dict[str, Any]]:
    """
    Chroma vector search를 수행합니다.

    Args:
        query_profile: build_query_profile 결과
        top_k: 반환 개수
        filter_dict: Chroma metadata filter

    Return:
        검색 결과 리스트
    """
    query = query_profile["expanded_query"]

    if filter_dict:
        docs_with_scores = vectorstore.similarity_search_with_relevance_scores(
            query=query,
            k=top_k,
            filter=filter_dict,
        )
    else:
        docs_with_scores = vectorstore.similarity_search_with_relevance_scores(
            query=query,
            k=top_k,
        )

    rows = []

    for rank, (doc, score) in enumerate(docs_with_scores, start=1):
        metadata = dict(doc.metadata)

        rows.append({
            "rank": rank,
            "chunk_id": metadata.get("chunk_id", ""),
            "score": float(score),
            "vector_score": float(score),
            "bm25_score": 0.0,
            "text": doc.page_content,
            "retrieval_method": "vector",
            **metadata,
        })

    return rows


test_profile = build_query_profile("금융광고 설명의무 고지")
test_vector_rows = vector_search(test_profile, top_k=5)

for row in test_vector_rows:
    print("=" * 100)
    print(
        row["rank"],
        row["law_name"],
        row["article_no"],
        row["article_title"],
        row["score"],
        row["retrieval_method"],
    )
    print(row["text"][:300])

1 금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402) 제17조 광고의 내용 0.5048278879072099 vector
[문서명: 금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402) / 조문: 제17조(광고의 내용) / 페이지: 17]
제17조(광고의 내용)
③ 금융상품판매업자등이 영 제18조제4항에 따라 일부 내용을 제외할 경우 준수해야 할 기준은 다음 각 호의 구
분에 따른다.
1. 보장성 상품에 관한 광고
가. 다음의 사항 전부 또는 일부만을 개괄적으로 알릴 것
　 1) 금융상품의 편익
　 2) 금융상품에 적합한 금융소비자의 특성 또는 가입요건
　 3) 금융상품의 특성
　 4) 판매채널의 특징 및 상담 연락처

2 금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402) 제19조 광고 시 금지행위 0.4995191551624748 vector
[문서명: 금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402) / 조문: 제19조(광고 시 금지행위) / 페이지: 18]
제19조(광고 시 금지행위)
① 영 제20조제1항제6호에서 "금융위원회가 정하여 고시하는 행위"란 다음 각 호의 구분
에 따른 행위를 말한다.
1. 금융소비자에 따라 달라질 수 있는 거래조건을 누구에게나 적용될 수 있는 것처럼 오인하게 만드는 행위
2. 보험금 지급사유나 지급시점이 다름에도 불구하고 각각의 보험금이 한꺼번에 지급되는 것처럼 오인하게 만드
는 행위
3. 금융상품에 
3 금융소비자 보호에 관한 법률(법률)(제21065호)(20260102) 제22조 금융상품등에 관한 광고 관련 준수사항 0.47932950817345266 vector
[문서명: 금융소비자 보호에 관한 법률(법률)(제21065호)(20260102) / 조문: 제22조(금융상품등에 관한 광고 관련 준수사항) / 페이지: 10]
제22조(금융상품등에 관한 광고 관련 준수사항)
④ 금융상품판매업

In [8]:
# BM25 search
def bm25_search(
    query_profile: Dict[str, Any],
    top_k: int = 10,
) -> List[Dict[str, Any]]:
    """
    BM25 keyword search를 수행합니다.
    score가 0 이하인 결과는 제거합니다.

    Args:
        query_profile: build_query_profile 결과
        top_k: 반환 개수

    Return:
        검색 결과 리스트
    """
    query = query_profile["expanded_query"]
    query_tokens = tokenize_for_bm25(query)

    if not query_tokens:
        return []

    scores = bm25.get_scores(query_tokens)
    ranked_indices = np.argsort(scores)[::-1]

    rows = []

    for idx in ranked_indices:
        idx = int(idx)
        score = float(scores[idx])

        if score <= 0:
            continue

        metadata = dict(bm25_metadatas[idx])

        rows.append({
            "rank": len(rows) + 1,
            "chunk_id": bm25_ids[idx],
            "score": score,
            "vector_score": 0.0,
            "bm25_score": score,
            "text": bm25_documents[idx],
            "retrieval_method": "bm25",
            **metadata,
        })

        if len(rows) >= top_k:
            break

    return rows


test_bm25_rows = bm25_search(build_query_profile("수수료"), top_k=5)

for row in test_bm25_rows:
    print("=" * 100)
    print(
        row["rank"],
        row["law_name"],
        row["article_no"],
        row["article_title"],
        round(row["bm25_score"], 4),
        row["retrieval_method"],
    )
    print(row["text"][:300])

1 금융소비자 보호에 관한 법률 시행령(대통령령)(제36287호)(20260428) 제37조 청약의 철회 22.1093 bm25
[문서명: 금융소비자 보호에 관한 법률 시행령(대통령령)(제36287호)(20260428) / 조문: 제37조(청약의 철회) / 페이지: 22]
제37조(청약의 철회)
⑥ 법 제46조제2항제2호다목에서 “수수료 등 대통령령으로 정하는 비용”이란 해당 금융상품 계약을 위해 금융상품
판매업자등이 제3자에게 이미 지급한 다음 각 호의 비용을 말한다.<개정 2022. 12. 8.>
1. 인지세 등 제세공과금
2. 저당권 설정 등에 따른 등기 비용
3. 그 밖에 제1호 및 제2호의 비용에 준하는 것으로서 금융위원회가 정하여 고시하는 비용
2 금융소비자 보호에 관한 법률 시행령(대통령령)(제36287호)(20260428) 제29조 금융상품 비교공시 14.6462 bm25
[문서명: 금융소비자 보호에 관한 법률 시행령(대통령령)(제36287호)(20260428) / 조문: 제29조(금융상품 비교공시) / 페이지: 19]
제29조(금융상품 비교공시)
② 금융상품의 비교공시에는 다음 각 호의 사항이 포함되어야 한다.
1. 이자율
2. 보험료
3. 수수료
4. 그 밖에 금융소비자가 금융상품의 계약 체결 여부를 판단하는데 필요한 정보로서 금융위원회가 정하여 고시하는
사항
3 금융소비자 보호에 관한 법률 시행령(대통령령)(제36287호)(20260428) 제13조 설명의무 14.4811 bm25
[문서명: 금융소비자 보호에 관한 법률 시행령(대통령령)(제36287호)(20260428) / 조문: 제13조(설명의무) / 페이지: 9]
제13조(설명의무)
④ 법 제19조제1항제1호나목4)에서 “대통령령으로 정하는 사항”이란 다음 각 호의 사항(연계투자는 제4호만 해당
한다)을 말한다.
1. 금융소비자가 부담해야 하는 수수료
2. 계약의 해지ㆍ해제
3. 증권의 환매(還買) 및 매매
4. 「온라인투자연계금융업 및 이용자 보호에 관한 법률」 제22조제1항 각 호의

In [9]:
# RRF merge
def reciprocal_rank_fusion(
    result_lists: List[List[Dict[str, Any]]],
    k: int = 60,
) -> List[Dict[str, Any]]:
    """
    여러 검색 결과를 RRF 방식으로 병합합니다.

    Args:
        result_lists: 검색 결과 리스트들의 리스트
        k: RRF 보정 상수

    Return:
        병합된 검색 결과 리스트
    """
    scores = {}
    items = {}
    methods = defaultdict(set)
    vector_scores = defaultdict(float)
    bm25_scores = defaultdict(float)

    for result_list in result_lists:
        for rank, item in enumerate(result_list, start=1):
            chunk_id = item.get("chunk_id", "")

            if not chunk_id:
                continue

            scores[chunk_id] = scores.get(chunk_id, 0.0) + 1.0 / (k + rank)

            if chunk_id not in items:
                items[chunk_id] = dict(item)

            methods[chunk_id].add(item.get("retrieval_method", "unknown"))
            vector_scores[chunk_id] = max(vector_scores[chunk_id], safe_float(item.get("vector_score", 0.0)))
            bm25_scores[chunk_id] = max(bm25_scores[chunk_id], safe_float(item.get("bm25_score", 0.0)))

    merged = []

    for chunk_id, item in items.items():
        row = dict(item)
        row["rrf_score"] = scores[chunk_id]
        row["vector_score"] = vector_scores[chunk_id]
        row["bm25_score"] = bm25_scores[chunk_id]
        row["retrieval_method"] = "+".join(sorted(methods[chunk_id]))
        merged.append(row)

    return sorted(merged, key=lambda x: x["rrf_score"], reverse=True)

In [ ]:
# Deterministic rerank score
DOCUMENT_TYPE_PRIORITY = {
    "law": 1,
    "enforcement_decree": 2,
    "supervisory_regulation": 3,
    "guideline": 4,
    "faq_or_manual": 5,
    "unknown": 9,
}


def calculate_keyword_match_bonus(
    row: Dict[str, Any],
    preferred_keywords: List[str],
) -> float:
    """
    keyword 포함 여부에 따른 bonus를 계산합니다.

    Args:
        row: 검색 결과 row
        preferred_keywords: 선호 키워드 리스트

    Return:
        bonus 점수
    """
    if not preferred_keywords:
        return 0.0

    text = normalize_text(row.get("text", ""))
    metadata_keywords = split_pipe_string(row.get("keywords", ""))

    matched_count = 0

    for keyword in preferred_keywords:
        keyword_norm = normalize_text(keyword)

        if keyword_norm and keyword_norm in text:
            matched_count += 1
        elif keyword in metadata_keywords:
            matched_count += 1

    return min(matched_count * 0.02, 0.12)


def calculate_risk_tag_bonus(
    row: Dict[str, Any],
    preferred_risk_tags: List[str],
) -> float:
    """
    risk tag 일치 bonus를 계산합니다.

    Args:
        row: 검색 결과 row
        preferred_risk_tags: 선호 risk tag 리스트

    Return:
        bonus 점수
    """
    if not preferred_risk_tags:
        return 0.0

    row_tags = set(split_pipe_string(row.get("risk_tags", "")))
    preferred_tags = set(preferred_risk_tags)

    matched = row_tags.intersection(preferred_tags)

    return min(len(matched) * 0.05, 0.15)


def calculate_document_priority_bonus(row: Dict[str, Any]) -> float:
    """
    법령 위계 기반 bonus를 계산합니다.
    숫자가 낮은 document_priority가 상위 근거입니다.

    Args:
        row: 검색 결과 row

    Return:
        bonus 점수
    """
    document_type = row.get("document_type", "unknown")
    priority = DOCUMENT_TYPE_PRIORITY.get(document_type, 9)

    if priority == 1:
        return 0.05
    if priority == 2:
        return 0.04
    if priority == 3:
        return 0.03
    if priority == 4:
        return 0.01

    return 0.0


def apply_deterministic_rerank(
    rows: List[Dict[str, Any]],
    query_profile: Dict[str, Any],
) -> List[Dict[str, Any]]:
    """
    RRF 결과에 rule 기반 bonus를 반영해 최종 점수를 계산합니다.

    Args:
        rows: RRF 병합 결과
        query_profile: query profile

    Return:
        final_score 기준 정렬된 결과
    """
    reranked = []

    for row in rows:
        row = dict(row)

        keyword_bonus = calculate_keyword_match_bonus(
            row,
            query_profile.get("preferred_keywords", []),
        )
        risk_tag_bonus = calculate_risk_tag_bonus(
            row,
            query_profile.get("preferred_risk_tags", []),
        )
        document_priority_bonus = calculate_document_priority_bonus(row)

        method_bonus = 0.03 if row.get("retrieval_method") == "bm25+vector" else 0.0

        row["keyword_bonus"] = keyword_bonus
        row["risk_tag_bonus"] = risk_tag_bonus
        row["document_priority_bonus"] = document_priority_bonus
        row["method_bonus"] = method_bonus

        row["final_score"] = (
            safe_float(row.get("rrf_score", 0.0))
            + keyword_bonus
            + risk_tag_bonus
            + document_priority_bonus
            + method_bonus
        )

        reranked.append(row)

    return sorted(reranked, key=lambda x: x["final_score"], reverse=True)

In [11]:
# Parent expansion / dedupe
def attach_parent_context(row: Dict[str, Any]) -> Dict[str, Any]:
    """
    child 검색 결과에 parent 조문 전체 문맥을 붙입니다.

    Args:
        row: 검색 결과 row

    Return:
        parent_context가 추가된 row
    """
    row = dict(row)
    parent_id = row.get("parent_id", "")
    parent = parent_map.get(parent_id, {})

    row["parent_text"] = parent.get("text", "")
    row["parent_article_no"] = parent.get("article_no", row.get("article_no", ""))
    row["parent_article_title"] = parent.get("article_title", row.get("article_title", ""))

    return row


def dedupe_by_parent_id(
    rows: List[Dict[str, Any]],
    top_k: int = 5,
) -> List[Dict[str, Any]]:
    """
    parent_id 기준으로 중복 근거를 제거합니다.

    Args:
        rows: 검색 결과 리스트
        top_k: 최종 반환 개수

    Return:
        dedupe된 결과
    """
    deduped = []
    seen_parent_ids = set()

    for row in rows:
        parent_id = row.get("parent_id", "")

        if parent_id and parent_id in seen_parent_ids:
            continue

        if parent_id:
            seen_parent_ids.add(parent_id)

        deduped.append(row)

        if len(deduped) >= top_k:
            break

    return deduped

In [12]:
# 최종 hybrid_search 함수
def hybrid_search(
    query: str,
    vector_top_k: int = 15,
    bm25_top_k: int = 15,
    final_top_k: int = 5,
    filter_dict: Dict[str, Any] | None = None,
) -> List[Dict[str, Any]]:
    """
    query를 받아 vector + BM25 hybrid retrieval을 수행합니다.

    Args:
        query: 원문 질의
        vector_top_k: vector search 후보 수
        bm25_top_k: BM25 search 후보 수
        final_top_k: 최종 반환 개수
        filter_dict: Chroma metadata filter

    Return:
        최종 evidence 후보 리스트
    """
    query_profile = build_query_profile(query)

    vector_rows = vector_search(
        query_profile=query_profile,
        top_k=vector_top_k,
        filter_dict=filter_dict,
    )
    bm25_rows = bm25_search(
        query_profile=query_profile,
        top_k=bm25_top_k,
    )

    merged_rows = reciprocal_rank_fusion([vector_rows, bm25_rows])
    reranked_rows = apply_deterministic_rerank(merged_rows, query_profile)

    expanded_rows = [
        attach_parent_context(row)
        for row in reranked_rows
    ]

    final_rows = dedupe_by_parent_id(expanded_rows, top_k=final_top_k)

    return final_rows


test_hybrid_rows = hybrid_search("수수료 고지 설명의무", final_top_k=5)

for row in test_hybrid_rows:
    print("=" * 120)
    print(
        row["law_name"],
        row["article_no"],
        row["article_title"],
        "final:",
        round(row["final_score"], 4),
        "rrf:",
        round(row["rrf_score"], 4),
        "method:",
        row["retrieval_method"],
    )
    print("risk_tags:", row.get("risk_tags"))
    print("keywords:", row.get("keywords"))
    print(row["text"][:400])

금융소비자 보호에 관한 법률 시행령(대통령령)(제36287호)(20260428) 제13조 설명의무 final: 0.2159 rrf: 0.0159 method: bm25
risk_tags: explanation_duty|fee_missing|rate_condition_missing
keywords: 설명|설명의무|수수료|연|투자
[문서명: 금융소비자 보호에 관한 법률 시행령(대통령령)(제36287호)(20260428) / 조문: 제13조(설명의무) / 페이지: 9]
제13조(설명의무)
④ 법 제19조제1항제1호나목4)에서 “대통령령으로 정하는 사항”이란 다음 각 호의 사항(연계투자는 제4호만 해당
한다)을 말한다.
1. 금융소비자가 부담해야 하는 수수료
2. 계약의 해지ㆍ해제
3. 증권의 환매(還買) 및 매매
4. 「온라인투자연계금융업 및 이용자 보호에 관한 법률」 제22조제1항 각 호의 정보
5. 그 밖에 제1호부터 제4호까지의 사항에 준하는 것으로서 금융위원회가 정하여 고시하는 사항
금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402) 제13조 설명서 final: 0.1652 rrf: 0.0152 method: vector
risk_tags: advertising_regulation|approval_misleading|explanation_duty|fee_missing|principal_guarantee_misleading|rate_condition_missing
keywords: 금융투자|대출|보장|보험|보험금|보험료|비교|설명|손실|신용카드|연|연체
[문서명: 금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402) / 조문: 제13조(설명서) / 페이지: 10]
제13조(설명서)
① 금융상품판매업자등은 법 제19조제2항에 따른 설명서(이하 "설명서"라 한다)의 내용을 작성하는
경우에 다음 각 호의 사항을 준수하여야 한다.
1. 일반금융소비자가 쉽게 이해할 수 있도록 알기 쉬운 용어를 사용하여 

In [13]:
# Report-safe evidence 변환
def to_report_evidence(row: Dict[str, Any]) -> Dict[str, Any]:
    """
    검색 결과 row를 report/UI에 안전하게 노출할 evidence format으로 변환합니다.

    Args:
        row: 검색 결과 row

    Return:
        report-safe evidence dict
    """
    law_name = row.get("law_name", "")
    article_no = row.get("article_no", "")
    article_title = row.get("article_title", "")

    doc_title = f"{law_name} {article_no}({article_title})".strip()
    snippet = row.get("text", "").replace("\n", " ").strip()

    return {
        "doc_title": doc_title,
        "page": row.get("page", row.get("page_start", "")),
        "snippet": snippet[:500],
        "score": round(float(row.get("final_score", row.get("rrf_score", 0.0))), 5),
        "retrieval_method": row.get("retrieval_method", "hybrid"),
        # 내부 디버깅용. UI/report에서는 필요하면 숨길 수 있음.
        "chunk_id": row.get("chunk_id", ""),
        "parent_id": row.get("parent_id", ""),
        "document_type": row.get("document_type", ""),
        "risk_tags": row.get("risk_tags", ""),
    }


report_evidences = [
    to_report_evidence(row)
    for row in test_hybrid_rows
]

pprint(report_evidences)

[{'chunk_id': 'financial_consumer_act_enforcement_decree__article_13__child_004_2919d851',
  'doc_title': '금융소비자 보호에 관한 법률 시행령(대통령령)(제36287호)(20260428) 제13조(설명의무)',
  'document_type': 'enforcement_decree',
  'page': 9,
  'parent_id': 'financial_consumer_act_enforcement_decree__article_13',
  'retrieval_method': 'bm25',
  'risk_tags': 'explanation_duty|fee_missing|rate_condition_missing',
  'score': 0.21587,
  'snippet': '[문서명: 금융소비자 보호에 관한 법률 시행령(대통령령)(제36287호)(20260428) / 조문: '
             '제13조(설명의무) / 페이지: 9] 제13조(설명의무) ④ 법 제19조제1항제1호나목4)에서 “대통령령으로 정하는 '
             '사항”이란 다음 각 호의 사항(연계투자는 제4호만 해당 한다)을 말한다. 1. 금융소비자가 부담해야 하는 수수료 '
             '2. 계약의 해지ㆍ해제 3. 증권의 환매(還買) 및 매매 4. 「온라인투자연계금융업 및 이용자 보호에 관한 법률」 '
             '제22조제1항 각 호의 정보 5. 그 밖에 제1호부터 제4호까지의 사항에 준하는 것으로서 금융위원회가 정하여 '
             '고시하는 사항'},
 {'chunk_id': 'financial_consumer_supervisory_regulation__article_13__child_001_2c45ea57',
  'doc_title': '금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402) 제13조(설명서)',
  'docu

In [14]:
# Seed query smoke test
SEED_QUERIES = [
    "누구나 승인",
    "최저금리",
    "수수료",
    "원금보장",
    "확정수익",
    "설명의무",
    "부당권유",
    "금리",
    "이자율",
    "광고 오인",
]

seed_results = {}

for query in SEED_QUERIES:
    print("=" * 120)
    print("QUERY:", query)

    results = hybrid_search(query, final_top_k=5)
    seed_results[query] = results

    for rank, row in enumerate(results, start=1):
        print(
            f"{rank}. {row['law_name']} {row['article_no']}({row['article_title']}) "
            f"type={row['document_type']} "
            f"score={row['final_score']:.4f} "
            f"method={row['retrieval_method']}"
        )
        print("   risk_tags:", row.get("risk_tags"))
        print("   snippet:", row["text"][:180].replace("\n", " "))

QUERY: 누구나 승인
1. 금융소비자 보호에 관한 법률(법률)(제21065호)(20260102) 제22조(금융상품등에 관한 광고 관련 준수사항) type=law score=0.2441 method=vector
   risk_tags: advertising_regulation|approval_misleading|explanation_duty|fee_missing|principal_guarantee_misleading|rate_condition_missing|return_misleading
   snippet: [문서명: 금융소비자 보호에 관한 법률(법률)(제21065호)(20260102) / 조문: 제22조(금융상품등에 관한 광고 관련 준수사항) / 페이지: 10] 제22조(금융상품등에 관한 광고 관련 준수사항) ④ 금융상품판매업자등이 금융상품등에 관한 광고를 하는 경우 다음 각 호의 구분에 따른 행위를 해서는 아니 된다. 1
2. 금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402) 제17조(광고의 내용) type=supervisory_regulation score=0.1847 method=vector
   risk_tags: advertising_regulation|approval_misleading|principal_guarantee_misleading|rate_condition_missing
   snippet: [문서명: 금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402) / 조문: 제17조(광고의 내용) / 페이지: 17] 제17조(광고의 내용) ③ 금융상품판매업자등이 영 제18조제4항에 따라 일부 내용을 제외할 경우 준수해야 할 기준은 다음 각 호의 구 분에 따른다. 1. 보장성 상품에 관한
3. 예금자보호법(법률)(제21065호)(20260102) 제35조의4(개산지급금 지급의 승인) type=law score=0.1797 method=bm25+vector
   risk_tags

In [15]:
# “누구나 승인” 개선 확인
approval_results = hybrid_search("누구나 승인", final_top_k=10)

for rank, row in enumerate(approval_results, start=1):
    print("=" * 120)
    print(
        f"{rank}. {row['law_name']} {row['article_no']}({row['article_title']}) "
        f"type={row['document_type']} "
        f"score={row['final_score']:.4f} "
        f"method={row['retrieval_method']}"
    )
    print("risk_tags:", row.get("risk_tags"))
    print("keywords:", row.get("keywords"))
    print("text:", row["text"][:500].replace("\n", " "))

1. 금융소비자 보호에 관한 법률(법률)(제21065호)(20260102) 제22조(금융상품등에 관한 광고 관련 준수사항) type=law score=0.2441 method=vector
risk_tags: advertising_regulation|approval_misleading|explanation_duty|fee_missing|principal_guarantee_misleading|rate_condition_missing|return_misleading
keywords: 고지|광고|금리|대출|보장|보험|보험금|보험료|비용|설명|손실|수익
text: [문서명: 금융소비자 보호에 관한 법률(법률)(제21065호)(20260102) / 조문: 제22조(금융상품등에 관한 광고 관련 준수사항) / 페이지: 10] 제22조(금융상품등에 관한 광고 관련 준수사항) ④ 금융상품판매업자등이 금융상품등에 관한 광고를 하는 경우 다음 각 호의 구분에 따른 행위를 해서는 아니 된다. 1. 보장성 상품 가. 보장한도, 보장 제한 조건, 면책사항 또는 감액지급 사항 등을 빠뜨리거나 충분히 고지하지 아니하여 제한 없 이 보장을 받을 수 있는 것으로 오인하게 하는 행위 나. 보험금이 큰 특정 내용만을 강조하거나 고액 보장 사례 등을 소개하여 보장내용이 큰 것으로 오인하게 하는 행위 다. 보험료를 일(日) 단위로 표시하거나 보험료의 산출기준을 불충분하게 설명하는 등 보험료등이 저렴한 것으로 오인하게 하는 행위 라. 만기 시 자동갱신되는 보장성 상품의 경우 갱신 시 보험료등이 인상될 수 있음을 금융소비자가 인지할 수 있 도록 충분히 고지하지 아니하는 행위 마
2. 금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402) 제17조(광고의 내용) type=supervisory_regulation score=0.1847 method=vector
risk_tags: advertising_regulation|approval_misleading|principal_gu

In [16]:
# 간단 eval cases 정의
EVAL_CASES = [
    {
        "query": "누구나 승인",
        "expected_any_keywords": ["광고", "오인", "금지", "소비자"],
        "expected_any_risk_tags": ["approval_misleading", "advertising_regulation"],
        "preferred_doc_types": ["law", "enforcement_decree", "supervisory_regulation"],
    },
    {
        "query": "최저금리",
        "expected_any_keywords": ["금리", "이자율", "광고", "고지", "조건"],
        "expected_any_risk_tags": ["rate_condition_missing", "advertising_regulation", "explanation_duty"],
        "preferred_doc_types": ["law", "enforcement_decree", "supervisory_regulation"],
    },
    {
        "query": "수수료",
        "expected_any_keywords": ["수수료", "비용", "고지", "설명"],
        "expected_any_risk_tags": ["fee_missing", "explanation_duty"],
        "preferred_doc_types": ["law", "enforcement_decree", "supervisory_regulation"],
    },
    {
        "query": "원금보장",
        "expected_any_keywords": ["원금", "손실", "보장", "위험", "광고"],
        "expected_any_risk_tags": ["principal_guarantee_misleading", "advertising_regulation", "explanation_duty"],
        "preferred_doc_types": ["law", "enforcement_decree", "supervisory_regulation"],
    },
    {
        "query": "설명의무",
        "expected_any_keywords": ["설명", "설명의무", "중요사항", "고지"],
        "expected_any_risk_tags": ["explanation_duty"],
        "preferred_doc_types": ["law", "enforcement_decree", "supervisory_regulation"],
    },
]

In [17]:
# Eval runner
def evaluate_single_case(case: Dict[str, Any], top_k: int = 5) -> Dict[str, Any]:
    """
    단일 eval case에 대해 retrieval 품질을 간단히 평가합니다.

    Args:
        case: eval case dict
        top_k: 검색 결과 개수

    Return:
        평가 결과 dict
    """
    query = case["query"]
    results = hybrid_search(query, final_top_k=top_k)

    combined_text = " ".join(
        row.get("text", "")
        for row in results
    )

    combined_risk_tags = set()
    doc_types = set()

    for row in results:
        combined_risk_tags.update(split_pipe_string(row.get("risk_tags", "")))
        doc_types.add(row.get("document_type", ""))

    expected_keyword_hit = any(
        keyword in combined_text
        for keyword in case.get("expected_any_keywords", [])
    )

    expected_risk_tag_hit = any(
        tag in combined_risk_tags
        for tag in case.get("expected_any_risk_tags", [])
    )

    preferred_doc_type_hit = any(
        doc_type in doc_types
        for doc_type in case.get("preferred_doc_types", [])
    )

    return {
        "query": query,
        "result_count": len(results),
        "expected_keyword_hit": expected_keyword_hit,
        "expected_risk_tag_hit": expected_risk_tag_hit,
        "preferred_doc_type_hit": preferred_doc_type_hit,
        "top_docs": [
            {
                "doc_title": f"{row.get('law_name')} {row.get('article_no')}({row.get('article_title')})",
                "document_type": row.get("document_type"),
                "score": round(row.get("final_score", 0.0), 5),
                "method": row.get("retrieval_method"),
                "risk_tags": row.get("risk_tags"),
            }
            for row in results
        ],
    }


eval_results = [
    evaluate_single_case(case, top_k=5)
    for case in EVAL_CASES
]

for result in eval_results:
    print("=" * 120)
    pprint(result)

{'expected_keyword_hit': True,
 'expected_risk_tag_hit': True,
 'preferred_doc_type_hit': True,
 'query': '누구나 승인',
 'result_count': 5,
 'top_docs': [{'doc_title': '금융소비자 보호에 관한 법률(법률)(제21065호)(20260102) '
                            '제22조(금융상품등에 관한 광고 관련 준수사항)',
               'document_type': 'law',
               'method': 'vector',
               'risk_tags': 'advertising_regulation|approval_misleading|explanation_duty|fee_missing|principal_guarantee_misleading|rate_condition_missing|return_misleading',
               'score': 0.24408},
              {'doc_title': '금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402) '
                            '제17조(광고의 내용)',
               'document_type': 'supervisory_regulation',
               'method': 'vector',
               'risk_tags': 'advertising_regulation|approval_misleading|principal_guarantee_misleading|rate_condition_missing',
               'score': 0.18471},
              {'doc_title': '예금자보호법(법률)(제21065호)(20260102) 제35조의4(개산지급금 지급의

In [18]:
# Eval 요약표
df_eval = pd.DataFrame([
    {
        "query": row["query"],
        "result_count": row["result_count"],
        "expected_keyword_hit": row["expected_keyword_hit"],
        "expected_risk_tag_hit": row["expected_risk_tag_hit"],
        "preferred_doc_type_hit": row["preferred_doc_type_hit"],
        "pass": (
            row["result_count"] > 0
            and row["expected_keyword_hit"]
            and row["preferred_doc_type_hit"]
        ),
    }
    for row in eval_results
])

display(df_eval)

print("pass count:", int(df_eval["pass"].sum()), "/", len(df_eval))

,query,result_count,expected_keyword_hit,expected_risk_tag_hit,preferred_doc_type_hit,pass
0,누구나 승인,5,True,True,True,True
1,최저금리,5,True,True,True,True
2,수수료,5,True,True,True,True
3,원금보장,5,True,True,True,True
4,설명의무,5,True,True,True,True


pass count: 5 / 5


In [19]:
# Summary 저장
summary = {
    "collection_name": COLLECTION_NAME,
    "chroma_count": chroma_count,
    "bm25_documents_count": len(bm25_documents),
    "seed_queries": SEED_QUERIES,
    "eval_results": eval_results,
    "eval_pass_count": int(df_eval["pass"].sum()),
    "eval_total_count": len(df_eval),
}

with open(SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("summary 저장:", SUMMARY_PATH)
pprint(summary)

summary 저장: c:\Users\USER\Desktop\complypilot-jb\data\retrieval\debug_hybrid_retrieval\hybrid_retrieval_summary.json
{'bm25_documents_count': 2155,
 'chroma_count': 2155,
 'collection_name': 'complypilot_regulations_v2',
 'eval_pass_count': 5,
 'eval_results': [{'expected_keyword_hit': True,
                   'expected_risk_tag_hit': True,
                   'preferred_doc_type_hit': True,
                   'query': '누구나 승인',
                   'result_count': 5,
                   'top_docs': [{'doc_title': '금융소비자 보호에 관한 '
                                              '법률(법률)(제21065호)(20260102) '
                                              '제22조(금융상품등에 관한 광고 관련 준수사항)',
                                 'document_type': 'law',
                                 'method': 'vector',
                                 'risk_tags': 'advertising_regulation|approval_misleading|explanation_duty|fee_missing|principal_guarantee_misleading|rate_condition_missing|return_misleading',
              

In [20]:
# 최종 체크
print("=" * 100)
print("05_test_hybrid_retrieval 최종 체크")
print("=" * 100)

sample_hybrid = hybrid_search("광고 설명의무", final_top_k=3)
sample_evidence = [to_report_evidence(row) for row in sample_hybrid]

checks = {
    "chroma_count_gt_0": chroma_count > 0,
    "bm25_loaded": bm25 is not None,
    "parent_map_gt_0": len(parent_map) > 0,
    "hybrid_returns": len(sample_hybrid) > 0,
    "report_evidence_has_required_fields": all(
        all(key in ev for key in ["doc_title", "page", "snippet", "score", "retrieval_method"])
        for ev in sample_evidence
    ),
    "no_local_path_in_report_evidence": all(
        "c:\\" not in str(ev).lower()
        and "/users/" not in str(ev).lower()
        and "\\users\\" not in str(ev).lower()
        for ev in sample_evidence
    ),
    "eval_has_any_pass": int(df_eval["pass"].sum()) > 0,
}

pprint(checks)

if all(checks.values()):
    print("[OK] Hybrid retrieval 테스트 완료")
else:
    print("[WARN] 일부 체크가 실패했습니다. 결과를 보고 query/rerank 규칙을 보완하세요.")

05_test_hybrid_retrieval 최종 체크
{'bm25_loaded': True,
 'chroma_count_gt_0': True,
 'eval_has_any_pass': True,
 'hybrid_returns': True,
 'no_local_path_in_report_evidence': True,
 'parent_map_gt_0': True,
 'report_evidence_has_required_fields': True}
[OK] Hybrid retrieval 테스트 완료
